# Final Model Selection for Bike-Sharing Demand Prediction

After building and refining multiple regression models, the next step is to identify the best-performing model for deployment.

Model selection is not based solely on performance metrics, but also considers:

- Model accuracy
- Generalization capability
- Stability
- Interpretability

In this notebook, we:

- Compare baseline and refined models
- Evaluate regularized models (Ridge, Lasso, Elastic Net)
- Select the best model based on performance metrics
- Prepare the final model for deployment

This stage represents the transition from model experimentation to production readiness.

## Import Required Libraries

We use:

- `dplyr`: data manipulation  
- `caret`: model evaluation  
- `glmnet`: regularization models  

In [ ]:
# Load required libraries

# dplyr for data manipulation
library(dplyr)

# caret for data splitting and evaluation
library(caret)

# glmnet for regularized regression models
library(glmnet)

## Load Dataset and Prepare Data

In [ ]:
# Load dataset
bike_data <- read_csv("clean_bike_data.csv")

# Set seed for reproducibility
set.seed(123)

# Split dataset
train_index <- createDataPartition(bike_data$rented_bike_count, p = 0.8, list = FALSE)

train_data <- bike_data[train_index, ]
test_data <- bike_data[-train_index, ]

## Feature Engineering for Refined Models

In [ ]:
# Add polynomial and interaction features

train_data <- train_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2,
    temp_humidity = temperature * humidity
  )

test_data <- test_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2,
    temp_humidity = temperature * humidity
  )

## Train Candidate Models

We train multiple models:

- Baseline Linear Regression
- Refined Linear Regression
- Ridge Regression
- Lasso Regression
- Elastic Net

In [ ]:
# Baseline model
baseline_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed,
  data = train_data
)

# Refined model
refined_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed +
    temperature_sq + humidity_sq + temp_humidity,
  data = train_data
)

# Prepare matrices for glmnet
x_train <- model.matrix(rented_bike_count ~ ., train_data)[, -1]
y_train <- train_data$rented_bike_count

x_test <- model.matrix(rented_bike_count ~ ., test_data)[, -1]

# Ridge
ridge_model <- cv.glmnet(x_train, y_train, alpha = 0)

# Lasso
lasso_model <- cv.glmnet(x_train, y_train, alpha = 1)

# Elastic Net
elastic_model <- cv.glmnet(x_train, y_train, alpha = 0.5)

## Generate Predictions

In [ ]:
# Predictions

baseline_pred <- predict(baseline_model, newdata = test_data)

refined_pred <- predict(refined_model, newdata = test_data)

ridge_pred <- predict(ridge_model, s = "lambda.min", newx = x_test)

lasso_pred <- predict(lasso_model, s = "lambda.min", newx = x_test)

elastic_pred <- predict(elastic_model, s = "lambda.min", newx = x_test)

## Evaluate Model Performance

We compare models using:

- RMSE
- R-squared

In [ ]:
# RMSE function
rmse <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

# R2 function
r2 <- function(actual, predicted) {
  cor(actual, predicted)^2
}

# Compute metrics
results <- data.frame(
  Model = c("Baseline", "Refined", "Ridge", "Lasso", "ElasticNet"),
  RMSE = c(
    rmse(test_data$rented_bike_count, baseline_pred),
    rmse(test_data$rented_bike_count, refined_pred),
    rmse(test_data$rented_bike_count, ridge_pred),
    rmse(test_data$rented_bike_count, lasso_pred),
    rmse(test_data$rented_bike_count, elastic_pred)
  ),
  R2 = c(
    r2(test_data$rented_bike_count, baseline_pred),
    r2(test_data$rented_bike_count, refined_pred),
    r2(test_data$rented_bike_count, ridge_pred),
    r2(test_data$rented_bike_count, lasso_pred),
    r2(test_data$rented_bike_count, elastic_pred)
  )
)

# Display results
results

## Select Best Model

The best model is selected based on:

- Lowest RMSE
- Highest R-squared

Regularized models often provide better generalization performance.

In [ ]:
# Identify best model based on RMSE

best_model <- results[which.min(results$RMSE), ]

best_model

## Interpretation

From model comparison:

- Refined model improves over baseline by capturing non-linear relationships
- Regularized models reduce overfitting
- Elastic Net often balances bias and variance effectively

The selected model provides the best trade-off between accuracy and generalization.

## Preparing for Deployment

The selected model can be used in the prediction pipeline and integrated into the Shiny dashboard.

This ensures that predictions are generated consistently using the best-performing model. 

Note that the Lab had provided a pre trained model which is used in the Shiny App.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on comparing multiple regression models and selecting the best-performing model for deployment.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on regression model evaluation and selection.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the final model selection stage in the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Baseline Model Development  
- Model Refinement  
- Model Evaluation & Diagnostics  
- Feature Importance Analysis  
- **Final Model Selection**  
- Deployment (R Shiny Dashboard)

---

## Notes

Selecting the optimal model ensures that the prediction system achieves high accuracy while maintaining robustness and generalization capability.

---